# Relational Ranking LSTM (RRLSTM)
This section implements the Relational-Ranking LSTM (RRLSTM), the final and most sophisticated model in the notebook. It builds upon the Rank-LSTM by incorporating explicit relational information between stocks.

* ReRaLSTMModel: This nn.Module defines the core architecture. It does not contain an LSTM layer itself; instead, it takes the pre-computed sequential embeddings (from the Rank-LSTM) as input for a given day. Its key component is a Temporal Graph Convolution (TGC). For each stock, the model calculates attention weights over all other stocks based on their pre-defined relationships (e.g., same industry). It then creates a context vector by taking a weighted sum of the embeddings of related stocks. This context vector is concatenated with the stock's own embedding to make a final, relation-aware prediction.

* ReRaLSTM: The wrapper class manages the data loading and training for the RRLSTM. It loads the pre-trained embeddings and the relation graph. The training process uses the same combined regression and ranking loss as the Rank-LSTM, but the model learns to leverage inter-stock relationships to improve its ranking and prediction accuracy.

## Calculate Relation Strength
The model calculates a strength metric $g(a_ji, e_j^t, e_i^t)$ representing the influence of stock j on stock i.

### Explicit Modeling (`in_pro=true`)
This method defines the interaction strength as a product of two interpretable components: **similarity** and **relation importance**.

- *Similarity*: The inner product ($\mathbf{e}_i^t)^T \mathbf{e}_j^t$ measures the similarity of the two stocks' current states. The intuition is that a relationship is more likely to be active if the companies are currently behaving similarly.

- *Relation Importance*: A small neural network $\phi(\mathbf{w}^T \mathbf{a}_{ji} + b)$ learns a global importance score for the specific type of relation(s) $\mathbf{a}_{ji}$ that exist between j and i.

The complete strength function is:
$$
g(\mathbf{a}_{ji}, \mathbf{e}_j^t, \mathbf{e}_i^t) = \underbrace{ (\mathbf{e}_i^t)^T \mathbf{e}_j^t }_{\text{Similarity}} \times \underbrace{ \phi(\mathbf{w}^T \mathbf{a}_{ji} + b) }_{\text{Relation Importance}}
$$

### Implicit Modeling (`in_pro=false`):

This method is a more "black-box" approach. It concatenates the embeddings of both stocks and their relation vector and feeds them into a single neural network layer to learn the interaction strength directly, without assuming a specific functional form like the explicit model.

$$
g(\mathbf{a}_{ji}, \mathbf{e}_j^t, \mathbf{e}_i^t) = \phi(\mathbf{w}^T [\mathbf{e}_i^t, \mathbf{e}_j^t, \mathbf{a}_{ji}]^T + b)
$$

## Aggregate Information
Once the strength g(...) is computed for all pairs, the model creates the relational context vector $\mathbf{\tilde{e}}_i^t$ for stock i by performing a weighted sum (a "convolution" step) over all other stocks j. The weights are normalized by the degree $d_j$ (the number of outgoing relations from stock j).
$$
\mathbf{\tilde{e}}_i^t = \sum_{j | \text{sum}(\mathbf{a}_{ji}) > 0} \frac{g(\mathbf{a}_{ji}, \mathbf{e}_j^t, \mathbf{e}_i^t)}{d_j} \mathbf{e}_j^t
$$

This vector $\mathbf{\tilde{e}}_i^t$ represents the aggregated influence of the entire market on stock i, filtered through the lens of their relationships.

## Final Score
Finally, the model combines the original information with the new relational context. It concatenates the stock's own sequential embedding $\mathbf{e}_i^t$ with its relational context vector $\mathbf{\tilde{e}}_i^t$ and passes this combined vector through a final fully-connected layer to produce the ranking score.
$$
\hat{f}_i^{(t+1)} = \mathbf{w}_{pred}^T [\mathbf{e}_i^t, \mathbf{\tilde{e}}_i^t]^T + b_{pred}
$$

In [1]:
import argparse
import copy
import numpy as np
import os
import sys
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from time import time
import math
import scipy.stats as sps
from sklearn.metrics import mean_squared_error, mean_absolute_error

sys.path.append(os.path.abspath('../../'))
from models.evaluate import evaluate
from models.data_loading import load_EOD_data, load_relation_data

In [2]:
seed = 123456789
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
leaky_relu = lambda x, alpha=0.2: torch.maximum(alpha * x, x)

In [4]:
class ReRaLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, rel_encoding_shape, inner_prod=False, flat=False):
        super(ReRaLSTMModel, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.inner_prod = inner_prod
        self.flat = flat
        
        # Relation weight layer
        self.rel_weight_layer = nn.Linear(rel_encoding_shape[-1], 1)
        
        if not inner_prod:
            # Head and tail weight layers for sum weight
            self.head_weight = nn.Linear(input_dim, 1)
            self.tail_weight = nn.Linear(input_dim, 1)
        
        # Final layers
        if flat:
            self.hidden_layer = nn.Linear(input_dim * 2, hidden_dim)
        
        self.prediction_layer = nn.Linear(
            hidden_dim if flat else input_dim * 2, 1
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier/Glorot uniform initialization"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, feature, relation, rel_mask):
        batch_size = feature.size(0)
        
        # Compute relation weights
        rel_weight = F.leaky_relu(self.rel_weight_layer(relation), 0.2)
        
        if self.inner_prod:
            # Inner product weight
            inner_weight = torch.matmul(feature, feature.transpose(0, 1))
            weight = inner_weight * rel_weight[:, :, -1]
        else:
            # Sum weight
            head_weight = F.leaky_relu(self.head_weight(feature), 0.2)
            tail_weight = F.leaky_relu(self.tail_weight(feature), 0.2)
            
            all_one = torch.ones(batch_size, 1, device=feature.device)
            weight = (torch.matmul(head_weight, all_one.transpose(0, 1)) + 
                     torch.matmul(all_one, tail_weight.transpose(0, 1)) + 
                     rel_weight[:, :, -1])
        
        # Apply mask and softmax
        weight_masked = F.softmax(rel_mask + weight, dim=0)
        
        # Propagate features
        outputs_proped = torch.matmul(weight_masked.t(), feature)
        
        # Concatenate original and propagated features
        outputs_concated = torch.cat([feature, outputs_proped], dim=1)
        
        if self.flat:
            outputs_concated = F.leaky_relu(
                self.hidden_layer(outputs_concated), 0.2
            )

        predicted_return_ratio = self.prediction_layer(outputs_concated)
        
        return predicted_return_ratio


In [5]:
class ReRaLSTM:
    def __init__(self, data_path, market_name, tickers_fname, relation_name,
                 emb_fname, parameters, steps=1, epochs=50, batch_size=None, 
                 flat=False, gpu=False, in_pro=False):

        # Set seeds
        seed = 123456789
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        self.data_path = data_path
        self.market_name = market_name
        self.tickers_fname = tickers_fname
        self.relation_name = relation_name
        
        # Load data
        self.tickers = np.genfromtxt(os.path.join(data_path, '..', tickers_fname),
                                     dtype=str, delimiter='\t', skip_header=False)
        print('#tickers selected:', len(self.tickers))

        # Load the RAW prices for loss calculation
        raw_price_path = os.path.join(data_path, f'{market_name}_raw_prices.npy')
        self.raw_price_data = np.load(raw_price_path)
        if self.market_name == 'NASDAQ':
            self.raw_price_data = self.raw_price_data[:, :-1]
        print('Raw prices shape:', self.raw_price_data.shape)

        # Loads Z-score normalized features
        _, self.mask_data, self.gt_data, _ = \
            load_EOD_data(data_path, market_name, self.tickers, self.raw_price_data, steps)
        
        # Relation data
        rname_tail = {'sector_industry': '_industry_relation.npy',
                      'wikidata': '_wiki_relation.npy'}

        self.rel_encoding, self.rel_mask = load_relation_data(
            os.path.join(self.data_path, '..', 'relation', 'relation', self.relation_name,
                         self.market_name + rname_tail[self.relation_name])
        )
        print('relation encoding shape:', self.rel_encoding.shape)
        print('relation mask shape:', self.rel_mask.shape)

        self.embedding = np.load(
            os.path.join(self.data_path, '..', 'pretrain', 'pretrain', emb_fname))
        print('embedding shape:', self.embedding.shape)

        # The number of days must match between embeddings, masks, gt, and raw prices
        assert self.embedding.shape[1] == self.mask_data.shape[1], "Shape mismatch: embedding and mask"
        assert self.embedding.shape[1] == self.gt_data.shape[1], "Shape mismatch: embedding and ground truth"
        assert self.embedding.shape[1] == self.raw_price_data.shape[1], "Shape mismatch: embedding and raw prices"
        
        self.parameters = copy.copy(parameters)
        self.steps = steps
        self.epochs = epochs
        self.flat = flat
        self.inner_prod = in_pro
        
        if batch_size is None:
            self.batch_size = len(self.tickers)
        else:
            self.batch_size = batch_size

        self.valid_index = 756
        self.test_index = 1008
        self.trade_dates = self.embedding.shape[1]
        self.fea_dim = 5

        # Set device
        self.device = torch.device('cuda' if gpu and torch.cuda.is_available() else 'cpu')
        print('device:', self.device)

    def get_batch(self, offset=None):
        if offset is None:
            offset = random.randrange(0, self.valid_index)
        seq_len = self.parameters['seq']
        mask_batch = self.mask_data[:, offset: offset + seq_len + self.steps]
        mask_batch = np.min(mask_batch, axis=1)

        base_price_batch = self.raw_price_data[:, offset + seq_len - 1]
        
        for i in range(len(mask_batch)):
            if base_price_batch[i] < 1e-8:
                mask_batch[i] = 0.0
        
        return (self.embedding[:, offset, :],
                np.expand_dims(mask_batch, axis=1),
                np.expand_dims(base_price_batch, axis=1),
                np.expand_dims(self.gt_data[:, offset + seq_len + self.steps - 1], axis=1))

    def train(self):
        # Create model
        self.model = ReRaLSTMModel(
            input_dim=self.parameters['unit'],
            hidden_dim=self.parameters['unit'],
            rel_encoding_shape=self.rel_encoding.shape,
            inner_prod=self.inner_prod,
            flat=self.flat
        ).to(self.device)

        # Convert numpy arrays to tensors
        rel_encoding_tensor = torch.FloatTensor(self.rel_encoding).to(self.device)
        rel_mask_tensor = torch.FloatTensor(self.rel_mask).to(self.device)

        # Optimizer
        optimizer = optim.Adam(self.model.parameters(), lr=self.parameters['lr'])

        # Initialize best results
        best_valid_pred = np.zeros(
            [len(self.tickers), self.test_index - self.valid_index],
            dtype=float
        )
        best_valid_gt = np.zeros(
            [len(self.tickers), self.test_index - self.valid_index],
            dtype=float
        )
        best_valid_mask = np.zeros(
            [len(self.tickers), self.test_index - self.valid_index],
            dtype=float
        )
        best_test_pred = np.zeros(
            [len(self.tickers), self.trade_dates - self.parameters['seq'] -
             self.test_index - self.steps + 1], dtype=float
        )
        best_test_gt = np.zeros(
            [len(self.tickers), self.trade_dates - self.parameters['seq'] -
             self.test_index - self.steps + 1], dtype=float
        )
        best_test_mask = np.zeros(
            [len(self.tickers), self.trade_dates - self.parameters['seq'] -
             self.test_index - self.steps + 1], dtype=float
        )
        best_valid_perf = {'mse': np.inf, 'mrrt': 0.0, 'btl': 0.0}
        best_test_perf = {'mse': np.inf, 'mrrt': 0.0, 'btl': 0.0}
        best_valid_loss = np.inf

        batch_offsets = np.arange(start=0, stop=self.valid_index, dtype=int)
        
        for i in range(self.epochs):
            t1 = time()
            np.random.shuffle(batch_offsets)
            self.model.train()
            
            tra_loss = 0.0
            tra_reg_loss = 0.0
            tra_rank_loss = 0.0
            
            for j in range(self.valid_index - self.parameters['seq'] - self.steps + 1):
                emb_batch, mask_batch, price_batch, gt_batch = self.get_batch(batch_offsets[j])
                
                # Convert to tensors
                feature = torch.FloatTensor(emb_batch).to(self.device)
                mask = torch.FloatTensor(mask_batch).to(self.device)
                ground_truth = torch.FloatTensor(gt_batch).to(self.device)
                # base_price = torch.FloatTensor(price_batch).to(self.device)
                
                optimizer.zero_grad()
                
                # Forward pass
                # The model's output is the return_ratio
                return_ratio = self.model(feature, rel_encoding_tensor, rel_mask_tensor)
                
                # Calculate return ratio
                # return_ratio = (prediction - base_price) / base_price
                
                # Regression loss
                reg_loss = F.mse_loss(return_ratio * mask, ground_truth * mask)
                
                # Ranking loss
                pre_pw_dif = return_ratio - return_ratio.t()
                gt_pw_dif = ground_truth - ground_truth.t()
                mask_pw = mask @ mask.t()
                rank_loss = torch.mean(F.relu(-(pre_pw_dif * gt_pw_dif) * mask_pw))
                
                # Total loss
                total_loss = reg_loss + self.parameters['alpha'] * rank_loss
                
                # Backward pass
                total_loss.backward()

                # Gradient clipping
                # torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                tra_loss += total_loss.item()
                tra_reg_loss += reg_loss.item()
                tra_rank_loss += rank_loss.item()
            
            num_batches = self.valid_index - self.parameters['seq'] - self.steps + 1
            print('Train Loss:', tra_loss / num_batches, 
                  tra_reg_loss / num_batches, tra_rank_loss / num_batches)

            # Validation
            self.model.eval()
            cur_valid_pred = np.zeros(
                [len(self.tickers), self.test_index - self.valid_index], dtype=float
            )
            cur_valid_gt = np.zeros(
                [len(self.tickers), self.test_index - self.valid_index], dtype=float
            )
            cur_valid_mask = np.zeros(
                [len(self.tickers), self.test_index - self.valid_index], dtype=float
            )
            
            val_loss = 0.0
            val_reg_loss = 0.0
            val_rank_loss = 0.0
            
            with torch.no_grad():
                for cur_offset in range(
                    self.valid_index - self.parameters['seq'] - self.steps + 1,
                    self.test_index - self.parameters['seq'] - self.steps + 1
                ):
                    emb_batch, mask_batch, price_batch, gt_batch = self.get_batch(cur_offset)
                    
                    feature = torch.FloatTensor(emb_batch).to(self.device)
                    mask = torch.FloatTensor(mask_batch).to(self.device)
                    ground_truth = torch.FloatTensor(gt_batch).to(self.device)
                    # base_price = torch.FloatTensor(price_batch).to(self.device)
                    
                    return_ratio = self.model(feature, rel_encoding_tensor, rel_mask_tensor)
                    # return_ratio = (prediction - base_price) / base_price
                    
                    # Calculate losses
                    reg_loss = F.mse_loss(return_ratio * mask, ground_truth * mask)
                    
                    pre_pw_dif = return_ratio - return_ratio.t()
                    gt_pw_dif = ground_truth - ground_truth.t()
                    mask_pw = mask @ mask.t()
                    rank_loss = torch.mean(F.relu(-(pre_pw_dif * gt_pw_dif) * mask_pw))
                    
                    total_loss = reg_loss + self.parameters['alpha'] * rank_loss
                    
                    val_loss += total_loss.item()
                    val_reg_loss += reg_loss.item()
                    val_rank_loss += rank_loss.item()
                    
                    # Store predictions
                    idx = cur_offset - (self.valid_index - self.parameters['seq'] - self.steps + 1)
                    cur_valid_pred[:, idx] = return_ratio.cpu().numpy()[:, 0]
                    cur_valid_gt[:, idx] = gt_batch[:, 0]
                    cur_valid_mask[:, idx] = mask_batch[:, 0]
            
            val_batches = self.test_index - self.valid_index
            print('Valid MSE:', val_loss / val_batches, 
                  val_reg_loss / val_batches, val_rank_loss / val_batches)
            
            cur_valid_perf = evaluate(cur_valid_pred, cur_valid_gt, cur_valid_mask)
            print('\t Valid performance:', cur_valid_perf)

            # Testing
            cur_test_pred = np.zeros(
                [len(self.tickers), self.trade_dates - self.test_index], dtype=float
            )
            cur_test_gt = np.zeros(
                [len(self.tickers), self.trade_dates - self.test_index], dtype=float
            )
            cur_test_mask = np.zeros(
                [len(self.tickers), self.trade_dates - self.test_index], dtype=float
            )
            
            test_loss = 0.0
            test_reg_loss = 0.0
            test_rank_loss = 0.0
            
            with torch.no_grad():
                for cur_offset in range(
                    self.test_index - self.parameters['seq'] - self.steps + 1,
                    self.trade_dates - self.parameters['seq'] - self.steps + 1
                ):
                    emb_batch, mask_batch, price_batch, gt_batch = self.get_batch(cur_offset)
                    
                    feature = torch.FloatTensor(emb_batch).to(self.device)
                    mask = torch.FloatTensor(mask_batch).to(self.device)
                    ground_truth = torch.FloatTensor(gt_batch).to(self.device)
                    # base_price = torch.FloatTensor(price_batch).to(self.device)
                    
                    return_ratio = self.model(feature, rel_encoding_tensor, rel_mask_tensor)
                    # return_ratio = (prediction - base_price) / base_price
                    
                    # Calculate losses
                    reg_loss = F.mse_loss(return_ratio * mask, ground_truth * mask)
                    
                    pre_pw_dif = return_ratio - return_ratio.t()
                    gt_pw_dif = ground_truth - ground_truth.t()
                    mask_pw = mask @ mask.t()
                    rank_loss = torch.mean(F.relu(-(pre_pw_dif * gt_pw_dif) * mask_pw))
                    
                    total_loss = reg_loss + self.parameters['alpha'] * rank_loss
                    
                    test_loss += total_loss.item()
                    test_reg_loss += reg_loss.item()
                    test_rank_loss += rank_loss.item()
                    
                    # Store predictions
                    idx = cur_offset - (self.test_index - self.parameters['seq'] - self.steps + 1)
                    cur_test_pred[:, idx] = return_ratio.cpu().numpy()[:, 0]
                    cur_test_gt[:, idx] = gt_batch[:, 0]
                    cur_test_mask[:, idx] = mask_batch[:, 0]
            
            test_batches = self.trade_dates - self.test_index
            print('Test MSE:', test_loss / test_batches,
                  test_reg_loss / test_batches, test_rank_loss / test_batches)
            
            cur_test_perf = evaluate(cur_test_pred, cur_test_gt, cur_test_mask)
            print('\t Test performance:', cur_test_perf)
            
            # Update best results
            if val_loss / val_batches < best_valid_loss:
                best_valid_loss = val_loss / val_batches
                best_valid_perf = copy.copy(cur_valid_perf)
                best_valid_gt = copy.copy(cur_valid_gt)
                best_valid_pred = copy.copy(cur_valid_pred)
                best_valid_mask = copy.copy(cur_valid_mask)
                best_test_perf = copy.copy(cur_test_perf)
                best_test_gt = copy.copy(cur_test_gt)
                best_test_pred = copy.copy(cur_test_pred)
                best_test_mask = copy.copy(cur_test_mask)
                print('Better valid loss:', best_valid_loss)
                # self.save_model(self.model, f'../../data/pretrain/pretrain/{self.market_name}_{self.relation_name}_reralstm_model.pt')
            
            t4 = time()
            print('epoch:', i, ('time: %.4f ' % (t4 - t1)))
        
        print('\nBest Valid performance:', best_valid_perf)
        print('\tBest Test performance:', best_test_perf)

        return (best_valid_pred, best_valid_gt, best_valid_mask, best_valid_perf,
                best_test_pred, best_test_gt, best_test_mask, best_test_perf)

    def update_model(self, parameters):
        for name, value in parameters.items():
            self.parameters[name] = value
        return True

    def save_model(self, model, path):
        # Save both the model state and the parameters needed to reconstruct it
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': self.parameters['unit'],
            'hidden_dim': self.parameters['unit'],
            'rel_encoding_shape': self.rel_encoding.shape,
            'inner_prod': self.inner_prod,
            'flat': self.flat
        }, path)
        print(f"Model saved to {path}")

    def load_model(self, path):
        # Load the saved data
        checkpoint = torch.load(path, map_location=self.device)
        
        # Recreate the model architecture
        self.model = ReRaLSTMModel(
            input_dim=checkpoint['input_dim'],
            hidden_dim=checkpoint['hidden_dim'],
            rel_encoding_shape=checkpoint['rel_encoding_shape'],
            inner_prod=checkpoint['inner_prod'],
            flat=checkpoint['flat']
        ).to(self.device)
        
        # Load the trained weights
        self.model.load_state_dict(checkpoint['model_state_dict'])
        
        # Set to evaluation mode
        self.model.eval()
        print(f"Model loaded from {path}")
        return self.model

    def predict(self, model, start=None):
        model.eval()
        
        test_pred = np.zeros([len(self.tickers), self.trade_dates - start], dtype=float)
        test_gt = np.zeros_like(test_pred)
        test_mask = np.zeros_like(test_pred)
        
        with torch.no_grad():
            for offset in range(start - self.parameters['seq'] - self.steps + 1,
                                self.trade_dates - self.parameters['seq'] - self.steps + 1):
                emb_batch, mask_batch, _, gt_batch = self.get_batch(offset)
                
                feature = torch.FloatTensor(emb_batch).to(self.device)
                mask = torch.FloatTensor(mask_batch).to(self.device)
                gt = torch.FloatTensor(gt_batch).to(self.device)
                # base_price = torch.FloatTensor(price_batch).to(self.device)

                return_ratio = self.model(feature, 
                                  torch.FloatTensor(self.rel_encoding).to(self.device),
                                  torch.FloatTensor(self.rel_mask).to(self.device))
                # return_ratio = (prediction - base_price) / base_price

                idx = offset - (start - self.parameters['seq'] - self.steps + 1)
                test_pred[:, idx] = return_ratio.cpu().numpy()[:, 0]
                test_gt[:, idx] = gt_batch[:, 0]
                test_mask[:, idx] = mask_batch[:, 0]
                
            performance = evaluate(test_pred, test_gt, test_mask)
                
            return (
                test_pred,
                test_gt,
                test_mask,
                performance
            )


### NASDAQ-Industry

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [7]:
parameters = {'seq': 16, 'unit': 64, 'lr': 0.001, 'alpha': 0.1}
data_path='../../data/2013-01-01'
market_name='NASDAQ'
tickers_fname='NASDAQ_tickers_qualify_dr-0.98_min-5_smooth.csv'
relation_name='sector_industry'
emb_fname='NASDAQ_rank_lstm_seq-16_unit-64_2.csv.npy'

In [8]:
RR_LSTM = ReRaLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    relation_name=relation_name,
    emb_fname=emb_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    flat=False,
    gpu=True,
    in_pro=False
)

valid_pred, valid_gt, valid_mask, valid_perf, test_pred, test_gt, test_mask, test_perf = RR_LSTM.train()

loaded_model = RR_LSTM.load_model(f'../../data/pretrain/pretrain/{market_name}_{relation_name}_reralstm_model.pt')

#tickers selected: 1026
Raw prices shape: (1026, 1245)
single EOD data shape: (1245, 6)
relation encoding shape: (1026, 1026, 97)
relation encoding shape: (1026, 1026, 97)
relation mask shape: (1026, 1026)
embedding shape: (1026, 1245, 64)
device: cuda
Train Loss: 0.0025431036032976997 0.002529691969893237 0.00013411634460705525
Valid MSE: 0.0005470667466209899 0.000540556865055964 6.509879172188535e-05
	 Valid performance: {'mse': np.float64(0.0005449854273393112), 'mrrt': np.float64(0.0066895755736395635), 'btl': np.float64(0.8449575786362402), 'btl5': np.float64(1.1226188373868353), 'btl10': np.float64(1.2841527033888265)}
Test MSE: 0.0004219213646260888 0.00041604581407373494 5.8755511755687634e-05
	 Test performance: {'mse': np.float64(0.00041735367685712615), 'mrrt': np.float64(0.011185547204421588), 'btl': np.float64(1.6918627558043227), 'btl5': np.float64(1.510327046332532), 'btl10': np.float64(1.4143700907236074)}
Better valid loss: 0.0005470667466209899
Model saved to ../../d

In [9]:
print('Prediction performance:', test_perf)

Prediction performance: {'mse': np.float64(0.00037734721517207085), 'mrrt': np.float64(0.022731053879396376), 'btl': np.float64(0.8567542355740443), 'btl5': np.float64(1.255681764337351), 'btl10': np.float64(1.204833611306095)}


### NASDAQ-Wikidata

In [10]:
relation_name='wikidata'

In [11]:
RR_LSTM = ReRaLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    relation_name=relation_name,
    emb_fname=emb_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    flat=False,
    gpu=True,
    in_pro=False
)

valid_pred, valid_gt, valid_mask, valid_perf, test_pred, test_gt, test_mask, test_perf = RR_LSTM.train()

loaded_model = RR_LSTM.load_model(f'../../data/pretrain/pretrain/{market_name}_{relation_name}_reralstm_model.pt')

#tickers selected: 1026
Raw prices shape: (1026, 1245)
single EOD data shape: (1245, 6)
relation encoding shape: (1026, 1026, 43)
relation encoding shape: (1026, 1026, 43)
relation mask shape: (1026, 1026)
embedding shape: (1026, 1245, 64)
device: cuda
Train Loss: 0.0008504343680829766 0.0008398992044267534 0.00010535162718214623
Valid MSE: 0.0005365129535863479 0.0005327334826385292 3.779469948096361e-05
	 Valid performance: {'mse': np.float64(0.0005370979528516395), 'mrrt': np.float64(0.031398909803400085), 'btl': np.float64(1.8165922472253442), 'btl5': np.float64(1.3972959024278688), 'btl10': np.float64(1.3755998455970255)}
Test MSE: 0.00044604269501542917 0.0004419975137099317 4.045182120588899e-05
	 Test performance: {'mse': np.float64(0.0004433869572740353), 'mrrt': np.float64(0.02960914166921125), 'btl': np.float64(1.6603691077034455), 'btl5': np.float64(1.111791456061475), 'btl10': np.float64(1.0372919169043584)}
Better valid loss: 0.0005365129535863479
Model saved to ../../dat

In [12]:
print('Prediction performance:', test_perf)

Prediction performance: {'mse': np.float64(0.0003775667896117637), 'mrrt': np.float64(0.03390798754378929), 'btl': np.float64(2.218595513753826), 'btl5': np.float64(1.3048797256837137), 'btl10': np.float64(1.0932286095776358)}


### NYSE-Industry

In [13]:
parameters = {'seq': 8, 'unit': 32, 'lr': 0.001, 'alpha': 10}
data_path='../../data/2013-01-01'
market_name='NYSE'
tickers_fname='NYSE_tickers_qualify_dr-0.98_min-5_smooth.csv'
relation_name='sector_industry'
emb_fname='NYSE_rank_lstm_seq-8_unit-32_0.csv.npy'

In [14]:
RR_LSTM = ReRaLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    relation_name=relation_name,
    emb_fname=emb_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    flat=False,
    gpu=True,
    in_pro=False
)

valid_pred, valid_gt, valid_mask, valid_perf, test_pred, test_gt, test_mask, test_perf = RR_LSTM.train()

loaded_model = RR_LSTM.load_model(f'../../data/pretrain/pretrain/{market_name}_{relation_name}_reralstm_model.pt')

#tickers selected: 1737
Raw prices shape: (1737, 1245)
single EOD data shape: (1245, 6)
relation encoding shape: (1737, 1737, 108)
relation encoding shape: (1737, 1737, 108)
relation mask shape: (1737, 1737)
embedding shape: (1737, 1245, 32)
device: cuda
Train Loss: 0.0007074611986709328 0.0003368186459256857 3.7064255318175594e-05
Valid MSE: 0.0004973777176928706 0.0003768774894856578 1.2050022808089413e-05
	 Valid performance: {'mse': np.float64(0.0003774647295010536), 'mrrt': np.float64(0.007691578426904635), 'btl': np.float64(1.4171540750539862), 'btl5': np.float64(1.1962223405425896), 'btl10': np.float64(1.2167573109290966)}
Test MSE: 0.0003276920991847567 0.00024251547407625116 8.517662538097015e-06
	 Test performance: {'mse': np.float64(0.00024301310037553851), 'mrrt': np.float64(0.007336474997619647), 'btl': np.float64(1.35191510997538), 'btl5': np.float64(1.0947486438904883), 'btl10': np.float64(1.080781827209647)}
Better valid loss: 0.0004973777176928706
Model saved to ../../

In [15]:
print('Prediction performance:', test_perf)

Prediction performance: {'mse': np.float64(0.00022562968158225887), 'mrrt': np.float64(0.019074480246918993), 'btl': np.float64(0.18668869448447367), 'btl5': np.float64(1.1105657828848055), 'btl10': np.float64(1.033792789933614)}


### NYSE-Wikidata

In [16]:
relation_name='wikidata'

In [17]:
RR_LSTM = ReRaLSTM(
    data_path=data_path,
    market_name=market_name,
    tickers_fname=tickers_fname,
    relation_name=relation_name,
    emb_fname=emb_fname,
    parameters=parameters,
    steps=1,
    epochs=50,
    batch_size=None,
    flat=False,
    gpu=True,
    in_pro=False
)

valid_pred, valid_gt, valid_mask, valid_perf, test_pred, test_gt, test_mask, test_perf = RR_LSTM.train()

loaded_model = RR_LSTM.load_model(f'../../data/pretrain/pretrain/{market_name}_{relation_name}_reralstm_model.pt')

#tickers selected: 1737
Raw prices shape: (1737, 1245)
single EOD data shape: (1245, 6)
relation encoding shape: (1737, 1737, 33)
relation encoding shape: (1737, 1737, 33)
relation mask shape: (1737, 1737)
embedding shape: (1737, 1245, 32)
device: cuda
Train Loss: 0.001077160043041172 0.0006178997769737977 4.5926026811981685e-05
Valid MSE: 0.0006917946874247198 0.0003814572542429983 3.10337433571505e-05
	 Valid performance: {'mse': np.float64(0.00038205162748733705), 'mrrt': np.float64(0.005958549428989372), 'btl': np.float64(1.1437080939431326), 'btl5': np.float64(0.8202741575325496), 'btl10': np.float64(0.8792686612112446)}
Test MSE: 0.00048004415140188604 0.00023801389798342216 2.4203025221347934e-05
	 Test performance: {'mse': np.float64(0.00023850228526755993), 'mrrt': np.float64(0.003432425436781326), 'btl': np.float64(-0.44919901140383445), 'btl5': np.float64(0.6959804177924515), 'btl10': np.float64(0.7975791672797632)}
Better valid loss: 0.0006917946874247198
Model saved to ../

In [18]:
print('Prediction performance:', test_perf)

Prediction performance: {'mse': np.float64(0.00022557162291100214), 'mrrt': np.float64(0.03320037709410364), 'btl': np.float64(1.618219104479067), 'btl5': np.float64(1.34511656757968), 'btl10': np.float64(1.2280972964261316)}
